In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
invoices = pd.read_excel("CRATES-2026\\DIST-DSUPMK\\OCTOBER\\INITIAL-DATA\\OCT-24.xlsx", sheet_name="Crate Movement Report")

In [3]:
invoices.columns

Index(['DATES', 'STATUS', 'A/C', 'Invoice No.', 'QTY'], dtype='object')

In [4]:
invoices.head()

,DATES,STATUS,A/C,Invoice No.,QTY
0,2026-10-22,Inv,55,55260411,41
1,2026-10-23,Inv,55,55260412,39
2,2026-10-24,Inv,55,55260413,51
3,2026-10-22,Inv,57,57260421,136
4,2026-10-23,Inv,57,57260422,136


In [5]:
invoices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2871 entries, 0 to 2870
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   DATES        2871 non-null   datetime64[ns]
 1   STATUS       2871 non-null   object        
 2   A/C          2871 non-null   int64         
 3   Invoice No.  2871 non-null   int64         
 4   QTY          2871 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(1)
memory usage: 112.3+ KB


In [6]:
try:
    credit_notes = pd.read_excel("CRATES-2026\\DIST-DSUPMK\\OCTOBER\\INITIAL-DATA\\OCT-24.xlsx", sheet_name="Sheet1")
except:
    credit_notes = pd.DataFrame()
    print("Credit note file not found")

In [7]:
credit_notes.columns

Index(['DATES', 'STATUS', 'A/C', 'Invoice No.', 'QTY'], dtype='object')

In [8]:
credit_notes.head()

,DATES,STATUS,A/C,Invoice No.,QTY
0,2026-10-22,CN,62,62260466,1
1,2026-10-22,CN,62,62260467,1
2,2026-10-23,CN,299,299260335,2
3,2026-10-22,CN,393,393260291,0
4,2026-10-22,CN,582,582260693,0


In [9]:
credit_notes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 656 entries, 0 to 655
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   DATES        656 non-null    datetime64[ns]
 1   STATUS       656 non-null    object        
 2   A/C          656 non-null    int64         
 3   Invoice No.  656 non-null    int64         
 4   QTY          656 non-null    int64         
dtypes: datetime64[ns](1), int64(3), object(1)
memory usage: 25.8+ KB


In [10]:
credit_notes.tail()

,DATES,STATUS,A/C,Invoice No.,QTY
651,2026-10-22,CN,7098,709826488,0
652,2026-10-22,CN,7098,709826489,0
653,2026-10-22,CN,7098,709826490,0
654,2026-10-22,CN,7098,709826491,0
655,2026-10-22,CN,7098,709826492,0


In [11]:
# Define numeric columns for processing
numeric_columns = ['QTY']

In [12]:
# Clean column names first
credit_notes.columns = credit_notes.columns.str.strip()
invoices.columns = invoices.columns.str.strip()

def normalize_account_code(value):
    if pd.isna(value):
        return pd.NA
    try:
        return str(int(float(value)))
    except (TypeError, ValueError):
        text = str(value).strip()
        return text[:-2] if text.endswith('.0') and text[:-2].isdigit() else text

# Ensure QTY is integer and normalize account codes
invoices['QTY'] = invoices['QTY'].astype(int)
credit_notes['QTY'] = credit_notes['QTY'].astype(int)
invoices['A/C'] = invoices['A/C'].apply(normalize_account_code)
credit_notes['A/C'] = credit_notes['A/C'].apply(normalize_account_code)

# Ensure DATES are datetime and normalized to date-only (ignore time part)
invoices['DATES'] = pd.to_datetime(invoices['DATES'], errors='coerce').dt.normalize()
credit_notes['DATES'] = pd.to_datetime(credit_notes['DATES'], errors='coerce').dt.normalize()

# Only consider credit notes with STATUS == 'CN'
credit_notes_cn = credit_notes[credit_notes['STATUS'] == 'CN']

# Remove invoice rows that have an exact match in credit notes (by DATE + A/C + QTY)
filtered_invoices = invoices.copy()
for idx, credit_row in credit_notes_cn.iterrows():
    match = (
        (filtered_invoices['DATES'] == credit_row['DATES']) &
        (filtered_invoices['A/C'] == credit_row['A/C']) &
        (filtered_invoices['QTY'] == credit_row['QTY'])
    )
    match_idx = filtered_invoices[match].index
    if not match_idx.empty:
        # Drop only the first matching invoice for this credit note
        filtered_invoices = filtered_invoices.drop(match_idx[0])

# Sum duplicates by date + account so daily inserts use total quantity
final_df = (
    filtered_invoices
    .groupby(['DATES', 'A/C'], as_index=False)['QTY']
    .sum()
    .sort_values(['DATES', 'A/C'])
    .reset_index(drop=True)
)

print(final_df)

          DATES   A/C  QTY
0    2026-10-22   100   62
1    2026-10-22  1000   97
2    2026-10-22  1003   53
3    2026-10-22  1008   16
4    2026-10-22  1009   51
...         ...   ...  ...
2228 2026-10-24   964   34
2229 2026-10-24   982   44
2230 2026-10-24   989   41
2231 2026-10-24   996    6
2232 2026-10-24   997   54

[2233 rows x 3 columns]


In [13]:
# Export to Excel
final_df.to_excel('CRATES-2026\\DIST-DSUPMK\\OCTOBER\\FINAL-OUTPUT\\OCT-24-CRATES-INVOICES.xlsx', index=False)

In [14]:
# final_df.to_excel(r'Z:\SALES TEAM-DATA ANALYSIS\DAILY CRATE SUPPLY\2026-CRATES\MARCH 2026 DAILY  SUPPLY FOR SMKTS AND DEPOTS.xlsx', sheet_name='Sheet1', index=False)

In [15]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

file_path = r"Z:\SALES TEAM-DATA ANALYSIS\DAILY CRATE SUPPLY\2026-CRATES\OCTOBER 2026 DAILY  SUPPLY FOR SMKTS AND DEPOTS.xlsx"

def normalize_account_code(value):
    if pd.isna(value):
        return pd.NA
    try:
        return str(int(float(value)))
    except (TypeError, ValueError):
        text = str(value).strip()
        return text[:-2] if text.endswith('.0') and text[:-2].isdigit() else text

# -------------------------
# LOAD TEMPLATE
# -------------------------
df = pd.read_excel(
    file_path,
    sheet_name="ALL CUSTOMERS",
    skiprows=2
)

df.columns = df.columns.str.strip()

# -------------------------
# PREPARE SOURCE DATA
# -------------------------
final_df.columns = final_df.columns.str.strip()
final_df['DATES'] = pd.to_datetime(final_df['DATES'], errors='coerce')
final_df['A/C'] = final_df['A/C'].apply(normalize_account_code)

# -------------------------
# DAY SUFFIX FUNCTION
# -------------------------
def day_suffix(day):
    if 11 <= day <= 13:
        return f"{day}th"
    return f"{day}{['th','st','nd','rd','th','th','th','th','th','th'][day % 10]}"

# -------------------------
# PREPARE DAILY DATA (SUM BY DATE + A/C)
# -------------------------
daily_data = (
    final_df
    .dropna(subset=['DATES'])
    .groupby(['DATES', 'A/C'], as_index=False)['QTY']
    .sum()
)
daily_data['day_col'] = daily_data['DATES'].dt.day.apply(day_suffix)

# day columns available in template
day_columns = [c for c in df.columns if c.endswith(('st', 'nd', 'rd', 'th'))]

# Use only days that exist in both source and template
source_days = sorted(
    [c for c in daily_data['day_col'].unique() if c in day_columns],
    key=lambda x: int(''.join(ch for ch in x if ch.isdigit()))
)

# -------------------------
# MATCH TEMPLATE TYPES
# -------------------------
df['CODE'] = df['CODE'].apply(normalize_account_code)

# Fill each date column separately
for day_col in source_days:
    day_slice = daily_data[daily_data['day_col'] == day_col].copy()

    # Aggregate all rows for the same account on this day
    day_totals = day_slice.groupby('A/C', as_index=False)['QTY'].sum()

    qty_map = dict(zip(day_totals['A/C'], day_totals['QTY']))
    df[day_col] = df['CODE'].map(qty_map).fillna(0)

# convert CODE back to numeric
df['CODE'] = pd.to_numeric(df['CODE'], errors='coerce')

# -------------------------
# OPEN WORKBOOK
# -------------------------
wb = load_workbook(file_path)
ws = wb["ALL CUSTOMERS"]

header_row = 3
headers = {cell.value: cell.column for cell in ws[header_row]}

# -------------------------
# WRITE ALL SOURCE DATE COLUMNS
# -------------------------
write_day_cols = [d for d in source_days if d in headers]

for r, row in df.iterrows():
    excel_row = r + header_row + 1

    ws.cell(
        row=excel_row,
        column=headers['CODE']
    ).value = row['CODE']

    for day_col in write_day_cols:
        ws.cell(
            row=excel_row,
            column=headers[day_col]
        ).value = row.get(day_col, 0)

# -------------------------
# COLOR RULE (all updated source days)
# -------------------------
orange_fill = PatternFill(
    start_color="FD4D08",
    end_color="FD4D08",
    fill_type="solid"
)

start_col = headers['1st']
color_day_cols = [d for d in write_day_cols if d in headers]

for day_col in color_day_cols:
    day_excel_col = headers[day_col]

    for r in range(header_row + 1, ws.max_row + 1):
        day_value = ws.cell(row=r, column=day_excel_col).value

        # if this day > 0 skip
        if day_value and day_value > 0:
            continue

        zero_cells = []

        # scan backwards from this updated day
        for c in range(day_excel_col, start_col - 1, -1):
            cell = ws.cell(row=r, column=c)

            if cell.value == 0 or cell.value is None:
                zero_cells.append(cell)
            else:
                break

        # color if 2+ consecutive zeros
        if len(zero_cells) >= 2:
            for z in zero_cells:
                z.fill = orange_fill

# -------------------------
# DETECT NEW CUSTOMERS FROM SOURCE
# -------------------------

# convert to numeric
df['CODE'] = pd.to_numeric(df['CODE'], errors='coerce')
final_df['A/C'] = pd.to_numeric(final_df['A/C'], errors='coerce')

existing_codes = set(df['CODE'].dropna().astype(int))

# unique accounts from source
source_accounts = set(final_df['A/C'].dropna().astype(int))

# new customers must:
# 1. not exist in template
# 2. follow your numbering (>=2355 and <3000)
new_accounts = [
    acc for acc in source_accounts
    if acc not in existing_codes and 2355 <= acc < 3000
]

new_accounts = sorted(new_accounts)

day_columns = [c for c in df.columns if c.endswith(('st','nd','rd','th'))]

for acc in new_accounts:
    new_row = {col: 0 for col in day_columns}

    new_row.update({
        'CODE': acc,
        'NAME': '',
        'CATEGORY': '',
        'ROUTE NO.': '',
        'ROUTE': ''
    })

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    print(f"New customer added: {acc}")

# -------------------------
# SAVE FILE
# -------------------------
wb.save(file_path)

if write_day_cols:
    print(f"Inserted crate data separately for days: {', '.join(write_day_cols)}")
else:
    print("No matching day columns found between source data and template.")

print("Crate data inserted and formatting applied successfully.")

New customer added: 2502
New customer added: 2506
Inserted crate data separately for days: 22nd, 23rd, 24th
Crate data inserted and formatting applied successfully.
